# Baseline (TF-IDF + classical ML)

In [1]:
import sys
from pathlib import Path

# Make the repo importable: the local checkout, or a fresh clone on Kaggle/Colab. Then pin versions that must match.
REPO_URL = "https://github.com/BohdanZasymovych/ai-hw3-ai-text-detection.git"
REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "setup_env.py").exists()), None)
if REPO is None:
    REPO = Path("/kaggle/working/hw3" if Path("/kaggle").exists() else "/content/hw3")
    if REPO.exists():
        !git -C {REPO} pull -q
    else:
        !git clone -q --depth 1 {REPO_URL} {REPO}
sys.path.insert(0, str(REPO))

import setup_env

setup_env.install_pins()

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

from utils import (ARTIFACTS_DIR, DATA_DIR, ENV, LABEL_ORDER, SEED, cached, class_report, ensure_data, ensure_dirs,
                   get_split, get_subsample, group_f1, macro_f1, plot_confusion, plot_group_f1, submit_to_kaggle,
                   write_submission)

warnings.simplefilter("ignore", ConvergenceWarning)
ensure_dirs()
ensure_data()
split = get_split()  # fixed train/val split, made and cached on first use
print(f"env={ENV}  data={DATA_DIR}  artifacts={ARTIFACTS_DIR}")

SUBMIT = False  # True: predict the test set and upload the submission to Kaggle (5 uploads a day)


[cache] loaded /home/bohdan/code/ai/hw3-ai-generated-text-detection/artifacts/split.pkl
env=local  data=/home/bohdan/code/ai/hw3-ai-generated-text-detection/data  artifacts=/home/bohdan/code/ai/hw3-ai-generated-text-detection/artifacts


## Data

Load the fixed validation split and the fixed training subsample (the same ones all notebooks use), and the test set.

In [2]:
train_full = pd.read_csv(DATA_DIR / "train.csv")
subsample = get_subsample()
train = train_full.iloc[subsample].reset_index(drop=True)
val = train_full.iloc[split["val"]].reset_index(drop=True)
del train_full
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

pd.DataFrame({"texts": [len(train), len(val), len(test)]}, index=["train subsample", "validation", "test"])

[cache] loaded /home/bohdan/code/ai/hw3-ai-generated-text-detection/artifacts/split.pkl
[cache] saved /home/bohdan/code/ai/hw3-ai-generated-text-detection/artifacts/subsample.npy


,texts
train subsample,50000
validation,27238
test,25000


## Features

TF-IDF vectorizers for word n-grams (1-2) and character n-grams (3-5), fitted on the training subsample only. Matrices and vectorizers are cached.

In [ ]:
CHAR_LIMIT = 3_000  # character features use only the start of a text, to limit memory
data_key = {"n_train": len(train), "n_val": len(val), "checksum": int(subsample.sum()), "char_limit": CHAR_LIMIT, "v": 2}


def clip(texts, kind):
    return texts if kind == "word" else texts.str[:CHAR_LIMIT]


def make_vectorizer(kind):
    if kind == "word":
        return TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=200_000, sublinear_tf=True, dtype=np.float32)
    return TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_features=200_000,
                           sublinear_tf=True, lowercase=False, dtype=np.float32)


def get_vectorizer(kind):
    """TF-IDF vectorizer fitted on the training subsample only."""
    return cached(f"tfidf_{kind}_vectorizer", lambda: make_vectorizer(kind).fit(clip(train["text"], kind)),
                  key={**data_key, "kind": kind})


features = {}


def get_features(name):
    """Train rows followed by validation rows for "word", "char" or "word+char". Only one feature set is kept in memory."""
    if name not in features:
        features.clear()
        parts = []
        for kind in name.split("+"):
            vec = get_vectorizer(kind)
            parts.append(cached(f"tfidf_{kind}_matrices",
                                lambda: (vec.transform(clip(train["text"], kind)), vec.transform(clip(val["text"], kind))),
                                key={**data_key, "kind": kind}))
        X_train, X_val = (sp.hstack(m, format="csr") for m in zip(*parts))
        features[name] = sp.vstack([X_train, X_val], format="csr")
    return features[name]

## Models and Search

Two class-balanced linear models, with a small search over C for each of three feature sets (word, char, both). `GridSearchCV` runs the search with a `PredefinedSplit`, so the only fold is the fixed train/validation split and the score is macro-F1 on the validation rows. `refit=False`, because refitting would train on the validation rows. The result table of each feature set is cached.

In [ ]:
GRIDS = {"LinearSVC": [0.1, 0.3, 1], "LogisticRegression": [3, 10]}
MODELS = {
    "LinearSVC": LinearSVC(class_weight="balanced", random_state=SEED),
    "LogisticRegression": LogisticRegression(class_weight="balanced", max_iter=200, random_state=SEED),
}
param_grid = [{"clf": [MODELS[name]], "clf__C": grid} for name, grid in GRIDS.items()]
y_all = np.concatenate([train["label"], val["label"]])
fixed_split = PredefinedSplit([-1] * len(train) + [0] * len(val))  # -1: always in train, 0: the validation fold
label_order = np.array(LABEL_ORDER)


def run_search(feature_set):
    search = GridSearchCV(Pipeline([("clf", "passthrough")]), param_grid, scoring="f1_macro", cv=fixed_split, refit=False)
    search.fit(get_features(feature_set), y_all)
    return pd.DataFrame({
        "features": feature_set,
        "model": [type(p["clf"]).__name__ for p in search.cv_results_["params"]],
        "C": [p["clf__C"] for p in search.cv_results_["params"]],
        "macro_f1": search.cv_results_["mean_test_score"],
    })


tables = [cached(f"baseline_search_{feature_set}", lambda: run_search(feature_set),
                 key={**data_key, "features": feature_set, "grids": GRIDS, "seed": SEED})
          for feature_set in ["word", "char", "word+char"]]
results = pd.concat(tables).sort_values("macro_f1", ascending=False).reset_index(drop=True)
results.style.background_gradient(subset="macro_f1", cmap="Blues").format({"macro_f1": "{:.3f}"})

## Best Model

Refit the configuration with the highest validation macro-F1 on the training subsample and show its per-class scores.

In [ ]:
best = results.iloc[0]
best_config = {"features": best["features"], "model": best["model"], "C": float(best["C"]), "seed": SEED}


def fit_best():
    X = get_features(best["features"])
    model = clone(MODELS[best["model"]]).set_params(C=best["C"]).fit(X[:len(train)], train["label"])
    order = [list(model.classes_).index(label) for label in LABEL_ORDER]
    return {"model": model, "val_scores": model.decision_function(X[len(train):])[:, order]}


best_fit = cached("baseline_best", fit_best, key={**data_key, "best": best_config})
cached("baseline_val_scores", lambda: best_fit["val_scores"], key={**data_key, "best": best_config}, kind="npy")
val_pred = label_order[best_fit["val_scores"].argmax(axis=1)]
class_report(val["label"], val_pred).style.format({"precision": "{:.3f}", "recall": "{:.3f}", "f1": "{:.3f}", "support": "{:,.0f}"})

Confusion matrix. Colours show the share of each true class; the counts are in brackets.

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4))
plot_confusion(val["label"], val_pred, ax)
plt.show()

Macro-F1 per language and domain. The dashed line is the overall validation macro-F1.

In [ ]:
by_group = group_f1(val, val_pred, ["lang", "domain"])
fig, ax = plt.subplots(figsize=(7, 4))
plot_group_f1(by_group, best["macro_f1"], ax)
plt.tight_layout()
plt.show()

## Submission

Runs only when `SUBMIT` is True. Predict the test set with the best model, save a valid submission file and upload it to Kaggle. The upload is cached by the best configuration, so rerunning the notebook does not spend another daily submission.

In [ ]:
kinds = best["features"].split("+")


def predict_test():
    X_test = sp.hstack([get_vectorizer(kind).transform(clip(test["text"], kind)) for kind in kinds]).tocsr()
    model = best_fit["model"]
    order = [list(model.classes_).index(label) for label in LABEL_ORDER]
    return model.decision_function(X_test)[:, order]


if SUBMIT:
    test_scores = cached("baseline_test_scores", predict_test, key={**data_key, "best": best_config}, kind="npy")
    test_pred = label_order[test_scores.argmax(axis=1)]
    path = write_submission(test["id"], test_pred, sample_submission, "baseline")
    message = f"baseline: {best['features']} {best['model']} C={best['C']}, val macro-F1 {best['macro_f1']:.3f}"
    cached("baseline_upload", lambda: submit_to_kaggle(path, message), key={**data_key, "best": best_config}, kind="json")
    display(pd.Series(test_pred).value_counts(normalize=True).reindex(LABEL_ORDER).mul(100).round(1)
            .to_frame("share of test predictions, %").T)
else:
    print("SUBMIT is False: test prediction and upload skipped")
